In [ ]:
import pandas as pd
import numpy as np
import wbdata

DroppedColumns = ['indicator','country','unit','obs_status','decimal']

#World Bank datasets 
InequalityDatasets = ['SI.POV.GINI','SI.POV.DDAY','SI.DST.10TH.10','SI.DST.FRST.20','NE.TRD.GNFS.ZS','BX.KLT.DINV.WD.GD.ZS',
                      'NY.GDP.PCAP.KD','NY.GDP.MKTP.KD.ZG','NE.CON.PRVT.PC.KD',
                      'SL.UEM.TOTL.ZS','SL.UEM.1524.ZS','SL.TLF.CACT.ZS',
                      'FP.CPI.TOTL.ZG','GC.TAX.TOTL.GD.ZS','GC.XPN.TOTL.GD.ZS']

'''
all datasets have the same columns: (it is okay to drop)
{ indicator', 'country', 'countryiso3code', 'date', 'value', 'unit','obs_status', 'decimal'} 
'''
#get the datasets from all the differet countries and add them into a dictionary based on key value pairs

In [ ]:
InequalityDataDict = {}

for iterator in InequalityDatasets:
    RetrievedDataset = wbdata.get_data(iterator,date=('1990','2020'))           #iterate and retrieve indicator by indicator data
    IterDF = pd.DataFrame(RetrievedDataset)                                     #convert retrieved data into dataframe
    IterDF.drop(columns=DroppedColumns,inplace=True)                            #change the df so cols are isocode,date and value
    IterDF.rename(columns={'value':iterator},inplace=True)                      #renaming cols to name of indicator value 
    
    #IterDF[iterator] = IterDF[iterator].replace(np.nan,float(0))                #replace all empty entries with NA values
    IterDF['countryiso3code'] = IterDF['countryiso3code'].replace('',pd.NA)     #replaces all empty country values with pd.NA
    IterDF.dropna(subset='countryiso3code',inplace=True)                        #removes all NA entries from the dataframe 

    pd.to_numeric(IterDF[iterator],downcast='float')                            #this parses through all values and then converts them to floats

    InequalityDataDict.update({iterator:IterDF})

#so now all the data is in a dictionary of {IndicatorID : IndicatorDataFrame}

In [5]:
#merging of all indicators onto one dataframe
MasterDataframe = InequalityDataDict[InequalityDatasets[0]]

for iterator in range(1,len(InequalityDatasets)):
        CopyDf = InequalityDataDict[InequalityDatasets[iterator]] #this should now be a dataframe
        MasterDataframe = pd.merge(MasterDataframe,CopyDf,how='left',on=('countryiso3code','date'))

MasterDataframe.to_csv('RawDataFrame.csv')